# Семантический матчинг с помощью библиотеки sentence_transformers

В этом тюториале мы научимся семантическому поиску и ранжированию документов с помощью библиотеки _sentence_transformers_.

Документация библиотеки: https://www.sbert.net/

Ее можно поставить в виртуальное окружение командой `pip install sentence-transformers`

Импортируем модули которые нам понадобятся впоследствии:

In [1]:
import logging
from timeit import default_timer as timer

import numpy as np
import torch
import sentence_transformers as sbert

Подготовим логгер:

In [2]:
logging.basicConfig(level=logging.INFO)

## Данные

Будем работать с мини-сэмплом из пассажной части стандарного поискового датасета _MS MARCO_.

MS MARCO содержит:
- большую коллекцию документов, по которым можно искать
- реальные поисковые запросы из логов поисковика bing
- информацию о том, какие документы релевантны данному запросу

Подробнее про _MS MARCO_: https://microsoft.github.io/msmarco/

Возьмем несколько случайных запросов:

In [3]:
queries = [
    "how long is jupiter day",
    "what are isotopes of hydrogen",
]
queries

['how long is jupiter day', 'what are isotopes of hydrogen']

И несколько документов.

Выберем документы так, чтобы в нашем сэмпле были как релевантные запросам, так и нерелевантные, но "сложные": или содержащие ключевые слова, или просто близкие по тематике.

In [4]:
docs = [
    # About Jupiter
    "A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.", # relevant
    "Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral",
    "It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
    "The gas giant Jupiter has 63 known moons.",
    "Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun",
    "Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)",
    "Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.",
    "Io takes 1.77 Earth days to orbit Jupiter.",

    # About hydrogen
    "Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).", # relevant
    "A deuterium atom contains one proton, one neutron, and one electron.",
    "The fusion of deuterium with tritium creates helium-4, frees a remaining neutron, and releases energy.",
    "These different versions of hydrogen are called isotopes.",
    "Hydrogen is the only element where the isotopes are given specific names.",
    "The 2 H (or hydrogen-2) isotope is usually called deuterium",
    "Tritium (hydrogen-3 or 3H) is a weakly radioactive isotope of the element hydrogen",
    "All elements have at least some radioactive isotopes.",
    
    # Misc
    "The official language in Bangladesh is Bengali, though English is widely spoken.",
    "Grasshoppers are medium to large insects.",
    "Debian and Ubuntu are the most influential Linux distributions ever.",
    "Мoscow is the largest city in Russia with a population of 12.1 million people",
]

docs

['A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.',
 'Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral',
 "It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
 'The gas giant Jupiter has 63 known moons.',
 'Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun',
 'Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)',
 'Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.',
 'Io takes 1.77 Earth days to orbit Jupiter.',
 'Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).',
 'A deuterium atom contains one proton, one neutron, and one electron.',
 'The fusion of deuterium with tritium creates helium-4, frees a remaining neutron, and releases energy.',
 'These different versions of hydrogen are call

## Семантический поиск

Начнем с того, что попробуем поискать наши запросы в нашей коллекции документов.

Для этого нам надо:
- заэмбеддить все документы
- заэмбеддить запросы
- найти к каждому запросу ближайшие документы по косинусному расстонию

Для простоты мы не будем использовать какой-либо векторный индекс (типа _faiss_), и вместо приближенного поиска ближайших соседей (ANN) будем делать полный KNN.

Подготовим устройство:

In [5]:
device = torch.device("cuda:0")
logging.info(f"created device {device}")

INFO:root:created device cuda:0


Загрузим модели семантического поиска/матчинга.

Таких моделей существует множества, они могу отличаться:
- по качеству
- размеру и скорости
- поддержке разных языков и т.п.

Для выбора моделей рекомендую опираться на бенчмарки, напр. на MTEB:
- тут можно найти описание и ссылку на лидерборд: https://github.com/embeddings-benchmark/mteb
- немного про сам бенчмарк: https://huggingface.co/blog/mteb

Обычно, для русского языка в 2026 г. используют какую-либо модель из:
- семейства моделей E5 от Microsoft, напр. _multilingual-e5-large_
- новых Qwen3-Embedding, напр. _Qwen3-Embedding-8B_
- моделей BGE, напр. _bge-m3_
- моделей от Сбера, напр. _FRIDA_

Мы с вами дальше будем работать с английским языком, поэтому нам подойдут более легкие модели из семейства _MiniLM_.

In [6]:
# Which model?
model_name = "sentence-transformers/all-MiniLM-L6-v2"
#model_name = "intfloat/multilingual-e5-base"
#model_name = "Qwen/Qwen3-Embedding-0.6B" # requires prompt!

# Model params
model_kwargs = {}
model_kwargs['dtype'] = torch.float16

# Load model
model = sbert.SentenceTransformer(model_name, device=str(device), trust_remote_code=True, model_kwargs=model_kwargs)
num_params = sum(t.numel() for t in model.parameters())
logging.info(f"loaded model: model_name = {model_name} num_params = {num_params}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Tempor

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Re

Убедимся, что модель загрузилась на GPU с помощью утилиты _nvidia-smi_:

In [7]:
!nvidia-smi

Thu Apr 30 19:48:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 Ti     On  |   00000000:01:00.0  On |                  N/A |
|  0%   41C    P2             21W /  165W |     606MiB /  16380MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Заэмбеддим запросы:

In [8]:
start = timer()
query_embeds = model.encode(queries)
logging.info(f"embedded queries: query_embeds.shape = {query_embeds.shape} elapsed = {timer() - start:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:embedded queries: query_embeds.shape = (2, 384) elapsed = 0.208


И, аналогично, заэмбеддим документы:

In [9]:
start = timer()
doc_embeds = model.encode(docs)
logging.info(f"embedded docs: doc_embeds.shape = {doc_embeds.shape} elapsed = {timer() - start:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:embedded docs: doc_embeds.shape = (20, 384) elapsed = 0.011


Посчитаем скоры близости, как правило это просто косинус между векторами-эмбеддингами:

In [10]:
scores = model.similarity(query_embeds, doc_embeds).cpu().numpy()
print(scores)
logging.info(f"computed similarities: scores.shape = {scores.shape}")

INFO:root:computed similarities: scores.shape = (2, 20)


[[ 7.5928e-01  4.1699e-01  4.5996e-01  5.2148e-01  6.7969e-01  7.3145e-01
   7.6855e-01  7.1729e-01  5.5176e-02  1.2627e-02 -1.2856e-02  2.8336e-02
   9.8724e-03  3.6652e-02  1.2550e-02 -4.1901e-02  3.1433e-03  5.5054e-02
  -8.5693e-02 -6.1249e-02]
 [ 6.4453e-02  1.5320e-02  2.8564e-01  1.1688e-01  1.0730e-01  1.9543e-01
   2.1240e-02  8.5693e-02  8.1299e-01  5.4102e-01  4.1943e-01  8.6035e-01
   7.9248e-01  7.5049e-01  7.2266e-01  4.8560e-01  4.1656e-03  1.2012e-01
   4.7455e-02  6.7663e-04]]


Напишем функцию, которая ранжирует документы по запросу, и возвращает ранжированный список пар документ-скор:

In [11]:
def rank_docs(docs, query_scores):
    ranked_doc_ids = np.argsort(query_scores)[::-1]
    ranked_docs = [docs[i] for i in ranked_doc_ids]
    doc_scores = query_scores[ranked_doc_ids].tolist()
    docs_with_scores = list(zip(ranked_docs, doc_scores))
    return docs_with_scores

Применим к первому запросу про Юпитер:

In [12]:
print(f"Query: {queries[0]}")
rank_docs(docs, scores[0])

Query: how long is jupiter day


[('Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.',
  0.7685546875),
 ('A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.',
  0.75927734375),
 ('Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)',
  0.7314453125),
 ('Io takes 1.77 Earth days to orbit Jupiter.', 0.71728515625),
 ('Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun',
  0.6796875),
 ('The gas giant Jupiter has 63 known moons.', 0.521484375),
 ("It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
  0.4599609375),
 ('Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral',
  0.4169921875),
 ('Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).',
  0.05517578125),
 ('Grasshoppers are medium to large insects.', 0.0550537109375),
 ('Th

Видим, что ранжирование по близости эмбеддингов работает неплохо, но не идеально.

С одной стороны, документы про Юпитер действительно поднялись выше, что документы про водород. С другой стороны, на первое место вылез документы, содержащий все ключевые слова, но не отвечающий на вопрос пользователя.

Посмотрим теперь как ранжируются документы по второму запросу про водород:

In [14]:
print(f"Query: {queries[1]}")
rank_docs(docs, scores[1])

Query: what are isotopes of hydrogen


[('These different versions of hydrogen are called isotopes.', 0.8603515625),
 ('Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).',
  0.81298828125),
 ('Hydrogen is the only element where the isotopes are given specific names.',
  0.79248046875),
 ('The 2 H (or hydrogen-2) isotope is usually called deuterium',
  0.75048828125),
 ('Tritium (hydrogen-3 or 3H) is a weakly radioactive isotope of the element hydrogen',
  0.72265625),
 ('A deuterium atom contains one proton, one neutron, and one electron.',
  0.541015625),
 ('All elements have at least some radioactive isotopes.', 0.485595703125),
 ('The fusion of deuterium with tritium creates helium-4, frees a remaining neutron, and releases energy.',
  0.41943359375),
 ("It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
  0.28564453125),
 ('Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)',
  0.1954345703125),
 ('Grass

Тут у нас аналогичная картина -- ранжирование хорошее, но не идеальное.

## Ранжирование

Попробуем немного исправить ситуацию и улучшить качество ранжирования за счет использования моделей кросс-энкодеров (реранкеров).

На всякий случай вспомним:
- модели-эмбеддеры (их еще называют _би-энкодеры_) получают на вход по отдельности текст документа или запроса, и выдают эмбеддинги
- кросс-энкодеры получают на вход сразу два текста: запроса и документа, и выдают число (ранк), по которому можно ранжировать документы

Главный минусы кросс-энкодеров:
- их надо применять ко всем парам запрос-документ
- эмбеддинги документов невозможно предрасчитать заранее и сохранить в векторном индексе

Но есть и плюс: ранжирование получается гораздо более точным.

Загрузим модели, тут тоже будем использовать легкую модель _MiniLM_:

In [15]:
# Which model?
ce_model_name = "cross-encoder/ms-marco-MiniLM-L6-v2"
#ce_model_name = "Qwen/Qwen3-Reranker-0.6B"

# Load model
ce_model = sbert.CrossEncoder(ce_model_name, device=str(device), model_kwargs=model_kwargs)
logging.info(f"loaded cross-encoder model: ce_model_name = {ce_model_name}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
INFO:sentence_transformers.base.model:No modules.json found for cross-encoder/ms-marco-MiniLM-L6-v2, initializing a new CrossEncoder model.
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https:

Представим данные в виде пар запрос-документ, т.к. в этом формате их ожидает модель:

In [16]:
num_docs = len(docs)

# Create query/doc pairs
q0 = [queries[0]] * num_docs
pairs0 = list(zip(q0, docs))
pairs0

[('how long is jupiter day',
  'A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.'),
 ('how long is jupiter day',
  'Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral'),
 ('how long is jupiter day',
  "It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium."),
 ('how long is jupiter day', 'The gas giant Jupiter has 63 known moons.'),
 ('how long is jupiter day',
  'Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun'),
 ('how long is jupiter day',
  'Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)'),
 ('how long is jupiter day',
  'Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.'),
 ('how long is jupiter day', 'Io takes 1.77 Earth days to orbit Jupiter.'),
 ('how long is jupiter day',
  'Hydrogen has three isotopes: hydrogen-1 (protium), hydr

Аналогично для второго запроса:

In [17]:
q1 = [queries[1]] * num_docs
pairs1 = list(zip(q1, docs))
pairs1

[('what are isotopes of hydrogen',
  'A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.'),
 ('what are isotopes of hydrogen',
  'Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral'),
 ('what are isotopes of hydrogen',
  "It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium."),
 ('what are isotopes of hydrogen',
  'The gas giant Jupiter has 63 known moons.'),
 ('what are isotopes of hydrogen',
  'Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun'),
 ('what are isotopes of hydrogen',
  'Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)'),
 ('what are isotopes of hydrogen',
  'Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.'),
 ('what are isotopes of hydrogen',
  'Io takes 1.77 Earth days to orbit Jupiter.'),
 ('what are isotopes of hydrogen',


Применим наши кросс-энкодеры к парам запрос-документ:

In [18]:
start = timer()
scores0 = ce_model.predict(pairs0)
print(scores0)
logging.info(f"reranked: query = {queries[0]} elapsed = {timer() - start:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:reranked: query = how long is jupiter day elapsed = 0.031


[  8.7109375  -5.1914062  -9.7890625  -7.4414062   5.7382812   6.0117188
   2.8652344   3.1972656 -11.2109375 -11.28125   -11.2265625 -11.34375
 -11.34375   -11.2890625 -11.15625   -11.328125  -11.2734375 -11.3046875
 -11.359375  -11.359375 ]


Аналогично и для второго запроса:

In [19]:
start = timer()
scores1 = ce_model.predict(pairs1)
print(scores1)
logging.info(f"reranked: query = {queries[1]} elapsed = {timer() - start:.3f}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:reranked: query = what are isotopes of hydrogen elapsed = 0.014


[-11.234375  -11.296875  -10.109375  -11.3828125 -11.2890625 -11.2734375
 -11.3515625 -11.34375     8.5546875  -8.6953125  -9.703125    5.2148438
   4.0273438   4.6289062   4.7734375  -5.5351562 -11.2734375 -11.3125
 -11.375     -11.265625 ]


Посмотрим что получилось, сначала для первого запроса про Юпитер:

In [20]:
print(f"Query: {queries[0]}")
rank_docs(docs, scores0)

Query: how long is jupiter day


[('A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.',
  8.7109375),
 ('Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)',
  6.01171875),
 ('Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun',
  5.73828125),
 ('Io takes 1.77 Earth days to orbit Jupiter.', 3.197265625),
 ('Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.',
  2.865234375),
 ('Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral',
  -5.19140625),
 ('The gas giant Jupiter has 63 known moons.', -7.44140625),
 ("It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
  -9.7890625),
 ('Tritium (hydrogen-3 or 3H) is a weakly radioactive isotope of the element hydrogen',
  -11.15625),
 ('Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).

Результат стал гораздо лучше!

Теперь документы не просто разделились на релевантные и совсем не релевантные, но и внутри группы релевантных более релевантные оказались выше, чем менее релевантные.

И, аналогично, для второго:

In [21]:
print(f"Query: {queries[1]}")
rank_docs(docs, scores1)

Query: what are isotopes of hydrogen


[('Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).',
  8.5546875),
 ('These different versions of hydrogen are called isotopes.', 5.21484375),
 ('Tritium (hydrogen-3 or 3H) is a weakly radioactive isotope of the element hydrogen',
  4.7734375),
 ('The 2 H (or hydrogen-2) isotope is usually called deuterium', 4.62890625),
 ('Hydrogen is the only element where the isotopes are given specific names.',
  4.02734375),
 ('All elements have at least some radioactive isotopes.', -5.53515625),
 ('A deuterium atom contains one proton, one neutron, and one electron.',
  -8.6953125),
 ('The fusion of deuterium with tritium creates helium-4, frees a remaining neutron, and releases energy.',
  -9.703125),
 ("It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
  -10.109375),
 ('A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.',
  -11.234375),
 ('Мoscow is the la

Обратим внимание на значения ранков -- это больше не косинусное расстояние, а некая величина, не имеющая физического смысла.

Распространенная ошибка: подбирать какой-то порог на значение ранка, и фильровать по нему документы -- так делать нельзя!

Таким образом, мы научились:
- пользоваться библиотекой _sentence_transformers_
- эмбеддить запросы и документы
- считать скоры семантической близости
- переранжировать документы с помощью моделей кросс-энкодеров

Дальше для самостоятельного изучения рекомендуем:
- попробовать другие модели
- взять датасет побольше, напр. полный _MS MARCO_
- научиться считать метрики качества ранжирования и сравнить разные модели между собой